# Exploration de données : dataset [AMES](http://jse.amstat.org/v19n3/decock.pdf)

## Introduction

Dans ces travaux pratiques, nous allons nous intéresser à la compréhension d'un des datasets qui nous servira de terrain de jeu à certains points de la formation.

Ce dataset contient tous types de variables, a des valeurs manquantes, un nombre important de features et celles-ci ne sont pas forcément adaptées aux méthodes qu'on essaye de leur appliquer : il modélise bien un dataset que vous rencontrerez au jour le jour en faisant de la data science.

Aujourd'hui, nous allons suivre ces étapes :

1. poser le problème
2. obtenir les données
3. nettoyer et encoder les données
4. explorer les données
5. appliquer ce que l'exploration a appris

In [ ]:
import matplotlib.pyplot as plt
import numpy
import pandas
import scipy.stats
import seaborn

pandas.set_option("future.no_silent_downcasting", True)

## Définition du problème

La problématique d'intérêt dans cette analyse est la prédiction de prix de maisons en Iowa, aux États-Unis.

Pour que le problème soit bien posé, il faut établir dès le début de l'analyse quelles sont les sorties attendues et les mesures d'évaluation de ces sorties.

- *Définissez la sortie du modèle.*
- *Proposez une ou plusieurs métriques pour évaluer la qualité de ce que le modèle prédira.*
- *Quel est le problème qui se pose si on ne définit pas ces deux éléments en amont ?*

Répondez dans la cellule suivante, en commentaire.

In [ ]:
# Vos réponses ici, en commentaire

### Conseils supplémentaires

- La sortie : que cherche-t-on à prédire, et est-ce une classe ou une valeur continue ?
- Les métriques : quelles erreurs de régression connaissez-vous ? Une erreur de 20 000 $ pèse-t-elle autant sur une maison à 100 000 $ que sur une maison à 700 000 $ ?
- Le problème : que se passe-t-il si l'on choisit la métrique après avoir vu les résultats ?

### Solution

- La sortie du modèle est le prix de vente de la maison (`SalePrice`, en dollars) : c'est un problème de régression.
- Métriques : l'erreur absolue moyenne (MAE), en dollars, facile à lire ; la racine de l'erreur quadratique moyenne (RMSE), qui pénalise davantage les grosses erreurs ; et une erreur relative, car une erreur de 20 000 $ ne pèse pas pareil sur une maison à 100 000 $ et sur une maison à 700 000 $ : la RMSE calculée sur le logarithme du prix, la métrique de la compétition Kaggle d'où viennent ces données.
- Sans les définir en amont, on introduit un biais statistique : on pourrait choisir a posteriori les métriques qui maximisent nos résultats.

## Récupération des données

La cellule suivante télécharge les données dans le répertoire `ames/`. Celui-ci contient en particulier un fichier `train.csv` que vous pouvez utiliser comme ensemble d'entraînement et de validation, et un fichier `test.csv` qui simule une utilisation en production : on ne dispose pas des targets. Les variables du CSV sont décrites dans le fichier `ames/data_description.txt`, que la cellule télécharge aussi sur votre machine sous Colab.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh ames
!ls -l ames/
try:
  from google.colab import files

  files.download("ames/data_description.txt")
except ImportError:  # Hors de Colab : le fichier est dans l'explorateur, sous ames/
  pass

### Lecture des données

- *Lisez `ames/train.csv` et `ames/test.csv` dans les DataFrames `maisons` et `maisons_test` avec `pandas.read_csv`. La colonne d'index des données est nommée `Id` dans le CSV.*
- *Affichez le nom des colonnes ainsi que la dimension des données (nombre de lignes, nombre de colonnes).*
- *Affichez le début de `ames/data_description.txt`.*

**Documentation utile**

- [`pandas.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [`pandas.DataFrame.shape`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `pandas.read_csv(path, index_col="Id")` : la colonne `Id` devient l'index au lieu d'une variable.
- `df.columns` : les noms des colonnes ; `df.shape` : le couple (lignes, colonnes).
- `f.read()[:2000]` : les 2 000 premiers caractères d'un fichier ouvert par `open`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
maisons = pandas.read_csv(..., index_col=...)
maisons_test = ...

print(f"Colonnes : {', '.join(maisons.columns)}")
print(f"Forme de maisons : {...}")
print(f"Forme de maisons_test : {...}")

with open("ames/data_description.txt") as f:
  print(f.read()[...])
```

#### Solution

In [ ]:
maisons = pandas.read_csv("ames/train.csv", index_col="Id")
maisons_test = pandas.read_csv("ames/test.csv", index_col="Id")

print(f"Colonnes : {', '.join(maisons.columns)}")
print(f"Forme de maisons : {maisons.shape}")
print(f"Forme de maisons_test : {maisons_test.shape}")

# Le début de la description des variables
with open("ames/data_description.txt") as f:
  print(f.read()[:2000])

# Un aperçu des premières lignes
maisons.head()

## Extraction des variables

*Créez les variables suivantes :*

- *`train_X` qui contient toutes les colonnes de `maisons` sauf `SalePrice`*
- *`train_y` qui contient `SalePrice` de `maisons`*
- *`test_X` qui contient toutes les colonnes de `maisons_test`*

*Créez aussi la variable `all_X`, concaténation de `train_X` et `test_X`, qui nous permettra de tout prétraiter de manière homogène.*

*__Attention, utiliser `test_X` (et donc `all_X`) peut induire un biais statistique, il faut utiliser seulement `train_X` quand on utilise les données pour dériver une valeur !__*

Le cours recommande de mettre des données de côté sans les regarder ; pourquoi alors toucher à `test_X` ? Dans le cadre de ces données (une compétition Kaggle), le test n'a pas de cible : il ne sert qu'à produire les prédictions. On le regarde seulement pour savoir ce qu'il faudra savoir traiter (ses valeurs manquantes), toutes les valeurs de remplissage se calculent sur `train_X`, et l'on encode les deux tables ensemble pour qu'elles aient les mêmes colonnes : une catégorie qui n'existe que dans le test y devient une colonne, qui vaut 0 sur tout le train.

**Documentation utile**

- [`pandas.DataFrame.drop`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html)
- [`pandas.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `maisons.drop(columns="SalePrice")` : une copie sans la colonne.
- `maisons["SalePrice"]` : une colonne, sous forme de `Series`.
- `pandas.concat([a, b])` : empile les lignes de `a` et de `b`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
train_X = maisons.drop(...)
train_y = ...
test_X = ...
all_X = pandas.concat(...)

print(train_X.shape, train_y.shape, test_X.shape, all_X.shape)
```

### Solution

In [ ]:
train_X = maisons.drop(columns="SalePrice")
train_y = maisons["SalePrice"]
test_X = maisons_test
all_X = pandas.concat([train_X, test_X])

print("Forme des données d'entraînement :", train_X.shape, train_y.shape)
print("Forme des données de test :", test_X.shape)
print("Concaténation :", all_X.shape)

## Nettoyage des données

Cette étape, jointe avec l'exploration, passe par (au moins) 4 étapes :

- gérer les valeurs manquantes
- préprocessing (texte, image, …)
- standardisation
- transformation

Ici, nous n'aurons pas besoin de préprocessing spécifique, car les données sont principalement catégorielles ou continues, sans données textuelles ou d'images.

### Valeurs manquantes

Les différences de format, les conditions de récolte des données ainsi que beaucoup d'autres facteurs entraînent de nombreuses valeurs manquantes dans la plupart des datasets.

Dans ce dataset, `NA` signifie souvent l'absence (pas de piscine, pas de garage) : lisez `data_description.txt` pour distinguer une absence, qui devient une catégorie, d'une vraie valeur manquante, qu'il faut imputer.

Il est souvent intéressant de quantifier ce manque et de le pallier si cela est judicieux. Il est par exemple intéressant de noter que pour utiliser `sklearn`, il faut qu'aucune valeur ne manque pour que la quasi totalité des modèles fonctionne.

- *Trouvez le nombre total d'éléments manquants dans `train_X`, puis dans `test_X`.*
- *Calculez, pour chacune des deux tables, le pourcentage d'éléments manquants par colonne, en ne gardant que les colonnes auxquelles il en manque, triées pour afficher en premier celles auxquelles il en manque le plus. Nommez les résultats `manques_train` et `manques_test`.*
- *Affichez toutes les colonnes contenant au moins une valeur manquante (train set et test set confondus), puis celles qui n'en ont que dans le test set.*

**Documentation utile**

- [`pandas.DataFrame.isnull`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isnull.html)
- [`pandas.DataFrame.sum`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html)
- [`pandas.Series.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sort_values.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `df.isnull()` : un tableau de booléens, `True` là où une valeur manque.
- `.sum()` compte les `True` de chaque colonne, `.sum().sum()` ceux de toute la table ; `.mean() * 100` donne le pourcentage par colonne.
- `s[s > 0]` garde les éléments positifs d'une `Series` ; `s.sort_values(ascending=False)` la trie en décroissant.
- `set(manques_train.index)` : l'ensemble des colonnes à manques du train ; `|` en fait l'union, `-` la différence.
- Attendu : 7 829 valeurs manquantes dans `train_X`, 7 878 dans `test_X`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def colonnes_incompletes(df: pandas.DataFrame) -> pandas.Series:
  pourcentages = ...  # pourcentage de valeurs manquantes par colonne
  pourcentages = pourcentages[...]
  return pourcentages.sort_values(...)


print("Valeurs manquantes dans train_X :", train_X.isnull().sum().sum())
print("Valeurs manquantes dans test_X :", ...)

manques_train = colonnes_incompletes(train_X)
manques_test = ...
print(manques_train)
print(manques_test)

print(set(manques_train.index) | ...)
print(...)  # seulement dans le test set
```

#### Solution

In [ ]:
def colonnes_incompletes(df: pandas.DataFrame) -> pandas.Series:
  df_na: pandas.Series = df.isnull().mean() * 100
  df_na = df_na[df_na > 0]
  return df_na.sort_values(ascending=False)


def affiche_pourcentages(s: pandas.Series) -> None:
  print("\n".join([f"{nom:>20.20} {valeur:5.2f}%" for nom, valeur in s.items()]))


print(
  "Nombre de valeurs manquantes dans le training set :", train_X.isnull().sum().sum()
)
print("Nombre de valeurs manquantes dans le test set :", test_X.isnull().sum().sum())
print()

print("Pourcentage de valeurs manquantes par feature dans le training set :")
manques_train = colonnes_incompletes(train_X)
affiche_pourcentages(manques_train)
print()

print("Pourcentage de valeurs manquantes par feature dans le test set :")
manques_test = colonnes_incompletes(test_X)
affiche_pourcentages(manques_test)
print()

print(
  "Colonnes comprenant des valeurs manquantes dans le training set ou le test set :"
)
print(", ".join(sorted(set(manques_train.index) | set(manques_test.index))))
print()

print("Colonnes comprenant des valeurs manquantes seulement dans le test set :")
print(", ".join(sorted(set(manques_test.index) - set(manques_train.index))))

### Gestion des données manquantes

Il n'y a pas d'approche universelle du remplissage de valeur manquante. Chaque variable doit être traitée au cas par cas.

*En fonction de la documentation présente dans le fichier `ames/data_description.txt`, répartissez les colonnes qui ont des valeurs manquantes en groupes, un par méthode de remplissage, et remplissez-les avec `DataFrame.fillna` :*

- *`colonnes_moyenne` : la moyenne (ou la médiane) pour une variable réelle dont la valeur manque vraiment ;*
- *`colonnes_mode` : le mode (la valeur la plus présente) pour une variable catégorielle dont la valeur manque vraiment ;*
- *`colonnes_zero` : 0 pour une quantité qui manque parce que l'élément est absent (la surface d'un sous-sol inexistant) ;*
- *une valeur que dicte la documentation : `Functional` vaut `Typ` sauf mention contraire ;*
- *toutes les autres : la catégorie `"NA"`, qui dit l'absence (pas d'allée, pas de piscine).*

*Les valeurs de remplissage (moyenne, mode) se calculent sur `train_X` seul. À la fin, `all_X` ne doit plus avoir aucune valeur manquante.*

**Documentation utile**

- [`pandas.DataFrame.fillna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html)
- [`pandas.DataFrame.mean`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mean.html)
- [`pandas.DataFrame.mode`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mode.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `all_X.fillna(train_X[colonnes_moyenne].mean())` : passer une `Series` (nom de colonne → valeur) remplit chaque colonne avec sa propre valeur, calculée ici sur le train.
- `train_X[colonnes_mode].mode().iloc[0]` : `mode()` renvoie une table (une colonne peut avoir plusieurs modes) ; sa première ligne en garde un par colonne.
- Les colonnes qui n'ont de manques que dans le test (`MSZoning`, `Utilities`, `KitchenQual`…) ont une ou deux valeurs manquantes : de vraies valeurs manquantes.
- Les surfaces et les comptes de sous-sol et de garage (`BsmtFinSF1`, `GarageCars`…) manquent quand il n'y a pas de sous-sol ou de garage : 0.
- `all_X.fillna("NA")`, en dernier, remplit tout le reste.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
colonnes_moyenne = ["LotFrontage", ...]
all_X = all_X.fillna(train_X[colonnes_moyenne].mean())

colonnes_mode = ["MSZoning", ...]
all_X = all_X.fillna(train_X[colonnes_mode].mode().iloc[0])

colonnes_zero = ["GarageArea", ...]
all_X[colonnes_zero] = all_X[colonnes_zero].fillna(...)

all_X["Functional"] = ...
all_X = ...

print(all_X.isnull().sum().sum())  # 0 attendu
```

#### Solution

In [ ]:
# Remplissage avec la moyenne
colonnes_moyenne = ["LotFrontage", "GarageYrBlt"]
all_X = all_X.fillna(train_X[colonnes_moyenne].mean())

# Remplissage avec le mode
colonnes_mode = [
  "MSZoning",
  "Electrical",
  "KitchenQual",
  "Exterior1st",
  "Exterior2nd",
  "SaleType",
  "Utilities",
]
all_X = all_X.fillna(train_X[colonnes_mode].mode().iloc[0])

# Remplissage avec 0 : pas de sous-sol, de garage ou de placage
colonnes_zero = [
  "GarageArea",
  "GarageCars",
  "BsmtFinSF1",
  "BsmtFinSF2",
  "BsmtFullBath",
  "BsmtHalfBath",
  "BsmtUnfSF",
  "MasVnrArea",
  "TotalBsmtSF",
]
all_X[colonnes_zero] = all_X[colonnes_zero].fillna(0)

# Un remplissage spécifique, que dicte data_description.txt
all_X["Functional"] = all_X["Functional"].fillna("Typ")

# On donne à tous les autres NAs la valeur string NA, qui sera une catégorie
all_X = all_X.fillna("NA")

# On vérifie que l'on n'a rien oublié
print(all_X.isnull().sum().sum())

### Variables catégorielles, mais codées numériquement

Certaines catégorie telles que `MSSubClass` sont codées avec des codes numériques exprimant des catégories (cf. `data_description.txt`). Si on ne fait rien, cette variable va être traitée comme numérique alors que les catégories décrites ne sont pas même ordinales.

In [ ]:
# On transforme le codage numérique en string afin que ce soit traité comme
# une variable catégorielle
colonnes_en_categories = ["MSSubClass"]
all_X[colonnes_en_categories] = all_X[colonnes_en_categories].astype(str)

### Variables ordinales

Pour ne pas perdre l'information d'ordre des variables ordinales, il est important d'utiliser le label encoding vu en cours : chaque modalité devient un entier, dans l'ordre des modalités.

- *Label encodez les variables `BsmtCond` et `FireplaceQu` (de `NA`, absent, à `Ex`, excellent : voir `data_description.txt`) avec `DataFrame.replace`, puis convertissez-les en entiers.*

La solution traite de même toutes les variables ordinales du dataset : la suite part de son résultat.

**Documentation utile**

- [`pandas.DataFrame.replace`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html)
- [`pandas.DataFrame.astype`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `all_X.replace({"BsmtCond": {"NA": 0, "Po": 1}})` : un dictionnaire de remplacement par colonne, sous le nom de la colonne.
- Les deux variables ont la même échelle de qualité : `NA`, `Po`, `Fa`, `TA`, `Gd`, `Ex`.
- `all_X[colonnes].astype(int)` : la conversion en entiers, une fois les chaînes remplacées.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
echelle_qualite = dict(NA=0, Po=1, Fa=..., TA=..., Gd=..., Ex=...)
remplacements = dict(BsmtCond=echelle_qualite, FireplaceQu=...)
all_X = all_X.replace(remplacements)

colonnes_ordinales = list(remplacements)
all_X[colonnes_ordinales] = all_X[colonnes_ordinales].astype(...)
```

#### Solution

In [ ]:
# Création du dictionnaire de dictionnaire utilisé pour le remplacement des variables ordinales en entiers
# Key   : "Nom de la variable"
# Value :  Dictionnaire de remplacement des modalités de la variable
remplacements = dict(
  Alley=dict(NA=0, Grvl=1, Pave=2),
  BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
  BsmtFinType1=dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6),
  BsmtFinType2=dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6),
  Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
  LandSlope=dict(Sev=1, Mod=2, Gtl=3),
  LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
  PavedDrive=dict(NA=0, N=1, P=2, Y=3),
  Street=dict(Grvl=1, Pave=2),
  Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  BsmtCond=dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5),
)
# Toutes les variables suivantes peuvent utiliser le même dictionnaire de remplacement
colonnes_qualite = [
  "BsmtCond",
  "BsmtQual",
  "ExterCond",
  "ExterQual",
  "FireplaceQu",
  "GarageCond",
  "GarageQual",
  "HeatingQC",
  "KitchenQual",
  "PoolQC",
]
echelle_qualite = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
for colonne_qualite in colonnes_qualite:
  remplacements[colonne_qualite] = echelle_qualite

all_X = all_X.replace(remplacements)

# Cast explicite de ces variables en type entier
colonnes_ordinales = list(remplacements.keys())
all_X[colonnes_ordinales] = all_X[colonnes_ordinales].astype(int)

### One hot encoding des variables catégorielles

Maintenant que les variables ordinales sont traitées, il faut encore traiter les variables catégorielles (nominales) : le one-hot encoding de `pandas.get_dummies` remplace chacune par une colonne indicatrice par modalité.

- *Avant d'appeler `pandas.get_dummies`, prévoyez le nombre de colonnes qu'aura `all_X` : les colonnes numériques restent, et chaque colonne non numérique est remplacée par autant de colonnes qu'elle a de modalités (`nunique`).*
- *Appliquez la transformation à `all_X`, puis vérifiez votre prévision.*

**Documentation utile**

- [`pandas.DataFrame.select_dtypes`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html)
- [`pandas.DataFrame.nunique`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nunique.html)
- [`pandas.get_dummies`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `all_X.select_dtypes("number")` : les colonnes numériques ; `select_dtypes(exclude="number")` : les autres.
- `df.nunique()` : le nombre de modalités de chaque colonne ; `.sum()` les additionne.
- `df.shape[1]` : le nombre de colonnes.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
numeriques = all_X.select_dtypes(...)
categorielles = all_X.select_dtypes(exclude=...)
print("Colonnes prévues :", numeriques.shape[1] + ...)

all_X = pandas.get_dummies(...)
print("Colonnes obtenues :", ...)
```

#### Solution

In [ ]:
numeriques = all_X.select_dtypes("number")
categorielles = all_X.select_dtypes(exclude="number")
print("Forme avant :", all_X.shape)
print("Colonnes numériques :", numeriques.shape[1])
print("Colonnes non numériques :", categorielles.shape[1])
print("Colonnes prévues :", numeriques.shape[1] + categorielles.nunique().sum())

all_X = pandas.get_dummies(all_X)
print("Forme après :", all_X.shape)

Chaque modalité de chaque variable catégorielle a été transformée en feature. La matrice d'entrée est passée de sa taille originale de $\mathbb{R}^{2919 \times 79}$ à $\mathbb{R}^{2919 \times 244}$.

### Réassemblage

Maintenant que nous avons traité nos données, nous séparons `all_X` en `train_X` et `test_X` de nouveau, d'après leurs index (les `Id` des maisons).

In [ ]:
train_X = all_X.loc[train_X.index]
test_X = all_X.loc[test_X.index]
print(train_X.shape, test_X.shape)

### Contrôle des données préparées

- *Vérifiez que `train_X` et `test_X` ont les mêmes colonnes, et qu'aucune valeur n'y manque.*
- *Listez les colonnes indicatrices (celles qu'a créées `get_dummies`) qui valent 0 sur tout le train. D'où viennent-elles, et qu'un modèle entraîné sur le train pourra-t-il en apprendre ?*

**Documentation utile**

- [`pandas.Index.equals`](https://pandas.pydata.org/docs/reference/api/pandas.Index.equals.html)
- [`pandas.DataFrame.any`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.any.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `train_X.columns.equals(test_X.columns)` : `True` si les deux tables ont les mêmes colonnes, dans le même ordre.
- Les colonnes de `get_dummies` sont booléennes : `train_X.select_dtypes(bool)`.
- `df.any()` dit, pour chaque colonne, si elle contient au moins un `True` ; `~` en prend la négation.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print("Mêmes colonnes :", train_X.columns.equals(...))
print("Valeurs manquantes :", ..., ...)

indicatrices = train_X.select_dtypes(bool)
print(list(indicatrices.columns[...]))
```

#### Solution

In [ ]:
print("Mêmes colonnes :", train_X.columns.equals(test_X.columns))
print("Valeurs manquantes :", train_X.isnull().sum().sum(), test_X.isnull().sum().sum())

indicatrices = train_X.select_dtypes(bool)
print("Indicatrices à 0 sur tout le train :")
print(list(indicatrices.columns[~indicatrices.any()]))

Les deux tables ont les mêmes colonnes et aucune valeur manquante. Une seule indicatrice vaut 0 sur tout le train : `MSSubClass_150`, une modalité qui n'apparaît que dans le test. Elle n'existe que parce qu'on a encodé les deux tables ensemble ; constante sur le train, elle n'apprend rien à un modèle, qui traitera ces maisons du test comme si elles n'avaient aucune des modalités vues. En production, une modalité inconnue arrivera de même, sans qu'on puisse réencoder : le `OneHotEncoder(handle_unknown="ignore")` de scikit-learn, ajusté sur le train seul, la code par des zéros.

## Exploration des données

### Analyse de la variable de sortie

Les principales étapes de nettoyage des données étant finies, on peut commencer à analyser le dataset. Cette [page du manuel de seaborn](https://seaborn.pydata.org/tutorial/distributions.html#plotting-univariate-histograms) explique les principales techniques nécessaires.

- *Utilisez `seaborn` pour visualiser la distribution de la variable de sortie : un histogramme, avec une estimation de densité.*
- *Comment les prix sont-ils répartis ?*

**Documentation utile**

- [`seaborn.histplot`](https://seaborn.pydata.org/generated/seaborn.histplot.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `seaborn.histplot(train_y, kde=True)` : l'histogramme, et la courbe d'une estimation de densité par noyau (KDE).
- `plt.show()` affiche la figure.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
seaborn.histplot(..., kde=...)
plt.show()
```

#### Solution

In [ ]:
seaborn.histplot(train_y, kde=True)
plt.show()

La distribution est asymétrique : la plupart des ventes se font entre 100 000 et 250 000 $, et une longue queue s'étire vers les prix élevés, jusqu'à 755 000 $.

### Asymétrie de la variable de sortie

Une variable de sortie très asymétrique gêne beaucoup de méthodes : les prix élevés, rares, pèsent lourd dans une erreur quadratique, et une erreur de 20 000 $ y compte autant sur une maison bon marché que sur une maison chère. Le cours propose de transformer une telle variable (Box-Cox) ; pour un prix, la transformation usuelle est le logarithme.

- *Mesurez l'asymétrie (skewness) de `train_y` avec `Series.skew`.*
- *Appliquez `numpy.log1p` à `train_y`, et mesurez de nouveau l'asymétrie.*
- *Comparez l'histogramme et le diagramme quantile-quantile (QQ-plot, contre une loi normale) avant et après la transformation.*

**Documentation utile**

- [`pandas.Series.skew`](https://pandas.pydata.org/docs/reference/api/pandas.Series.skew.html)
- [`numpy.log1p`](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html)
- [`scipy.stats.probplot`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.probplot.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `train_y.skew()` : 0 pour une distribution symétrique, positif pour une queue vers la droite. Attendu : environ 1,88 avant la transformation, 0,12 après.
- `numpy.log1p(train_y)` : $\log(1 + y)$, défini même pour $y = 0$.
- `_, axes = plt.subplots(2, 2, figsize=(10, 8))` : une grille de 2 × 2 graphiques, `axes[ligne][colonne]`.
- `seaborn.histplot(y, kde=True, ax=ax)` trace dans le graphique `ax`.
- `scipy.stats.probplot(y, dist="norm", plot=ax)` trace les quantiles de `y` contre ceux d'une loi normale : une distribution normale donne une droite.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
train_y_log = numpy.log1p(...)
print("Asymétrie avant :", train_y.skew())
print("Asymétrie après :", ...)

_, axes = plt.subplots(2, 2, figsize=(10, 8))
for ligne, y in zip(axes, [train_y, train_y_log]):
  seaborn.histplot(y, kde=True, ax=ligne[0])
  scipy.stats.probplot(..., dist="norm", plot=ligne[1])
plt.tight_layout()
plt.show()
```

#### Solution

In [ ]:
train_y_log = numpy.log1p(train_y)
print(f"Asymétrie avant : {train_y.skew():.2f}")
print(f"Asymétrie après : {train_y_log.skew():.2f}")

_, axes = plt.subplots(2, 2, figsize=(10, 8))
for ligne, y, titre in zip(
  axes, [train_y, train_y_log], ["SalePrice", "log(1 + SalePrice)"]
):
  seaborn.histplot(y, kde=True, ax=ligne[0])
  ligne[0].set_title(titre)
  scipy.stats.probplot(y, dist="norm", plot=ligne[1])
  ligne[1].set_title(f"QQ-plot de {titre}")
plt.tight_layout()
plt.show()

L'asymétrie passe d'environ 1,88 à 0,12 : après le logarithme, l'histogramme est presque symétrique, et le QQ-plot presque une droite, alors qu'avant les prix élevés s'en écartaient nettement vers le haut.

Ce n'est pas la normalité de la cible qu'exigent les modèles : la régression linéaire suppose (pour ses tests et ses intervalles de confiance) des résidus normaux, et les arbres de décision ne supposent rien sur la cible. Prédire le logarithme du prix sert surtout à ce qu'une erreur compte en proportion du prix : la RMSE sur le log, c'est l'erreur relative de la première partie. Un modèle entraîné sur `train_y_log` prédit un log-prix, que `numpy.expm1` ramène en dollars.

### Variable catégorielle et prix

Une variable catégorielle, comme le quartier (`Neighborhood`), s'explore avec deux graphiques : le nombre de ventes de chaque modalité (count plot), et le prix moyen par modalité (diagramme en bâtons). Après `get_dummies`, `train_X` n'a plus de colonne `Neighborhood` : utilisez `maisons`.

- *Tracez le count plot de `Neighborhood`, et les bâtons du prix moyen par quartier, triés par prix moyen.*
- *Quels quartiers sont les plus chers ? Quelle confiance accorder au prix moyen d'un quartier qui compte peu de ventes ?*

**Documentation utile**

- [`seaborn : bar plots`](https://seaborn.pydata.org/tutorial/categorical.html#bar-plots)
- [`seaborn.countplot`](https://seaborn.pydata.org/generated/seaborn.countplot.html)
- [`seaborn.barplot`](https://seaborn.pydata.org/generated/seaborn.barplot.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Avec 25 quartiers, des barres horizontales gardent les noms lisibles : la variable catégorielle en `y`.
- `seaborn.countplot(data=maisons, y="Neighborhood")` compte les lignes de chaque modalité.
- `seaborn.barplot(data=maisons, x="SalePrice", y="Neighborhood")` trace le prix moyen de chaque modalité, avec son intervalle de confiance à 95 % (le trait noir).
- `order=` prend la liste des modalités dans l'ordre d'affichage : `maisons.groupby("Neighborhood")["SalePrice"].mean().sort_values().index`.
- `plt.subplots(1, 2, figsize=(12, 8), sharey=True)` : deux graphiques côte à côte, qui partagent l'axe des quartiers.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
ordre = maisons.groupby(...)["SalePrice"].mean().sort_values().index

_, (ax_ventes, ax_prix) = plt.subplots(1, 2, figsize=(12, 8), sharey=True)
seaborn.countplot(data=maisons, y=..., order=ordre, ax=ax_ventes)
seaborn.barplot(data=maisons, x=..., y="Neighborhood", order=..., ax=ax_prix)
plt.tight_layout()
plt.show()
```

#### Solution

In [ ]:
ordre = maisons.groupby("Neighborhood")["SalePrice"].mean().sort_values().index

_, (ax_ventes, ax_prix) = plt.subplots(1, 2, figsize=(12, 8), sharey=True)
seaborn.countplot(data=maisons, y="Neighborhood", order=ordre, ax=ax_ventes)
ax_ventes.set_title("Nombre de ventes")
seaborn.barplot(data=maisons, x="SalePrice", y="Neighborhood", order=ordre, ax=ax_prix)
ax_prix.set_title("Prix moyen")
plt.tight_layout()
plt.show()

NoRidge, NridgHt et StoneBr ont les prix moyens les plus élevés (plus de 300 000 $), MeadowV, IDOTRR et BrDale les plus bas (environ 100 000 $) : le quartier explique une bonne part du prix.

Les quartiers sont très inégalement représentés : 225 ventes à NAmes, 2 à Blueste. L'intervalle de confiance du prix moyen (le trait noir) s'élargit quand les ventes sont peu nombreuses : le prix moyen d'un quartier rare est peu fiable, et un modèle aura peu d'exemples pour en apprendre l'effet.

### Analyse des corrélations à la variable de sortie

Maintenant que nous avons une bonne idée de la forme de la variable de sortie, analysons les variables qui lui sont le plus corrélées.

- *Calculez la corrélation de chaque variable de `train_X` avec la variable de sortie, et triez-les.*
- *Affichez avec une heatmap la matrice de corrélation des dix variables les plus corrélées et des dix plus anti-corrélées, entre elles et avec la variable de sortie.*
- *Quelles variables sont les plus corrélées au prix ?*

**Documentation utile**

- [`pandas.DataFrame.corrwith`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corrwith.html)
- [`pandas.DataFrame.corr`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)
- [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `train_X.corrwith(train_y)` : la corrélation de chaque colonne avec `train_y` ; `.dropna()` retire celles des colonnes constantes (`NaN`).
- `s.sort_values(ascending=False)` ; `pandas.concat([s[:10], s[-10:]])` : les dix premières et les dix dernières valeurs.
- `train_X[colonnes].corr()` : la matrice de corrélation de ces colonnes ; `pandas.concat([s, m], axis=1)` y ajoute la colonne `s`.
- `seaborn.heatmap(m, vmin=-1, vmax=1, cmap="vlag")` : une échelle de couleurs centrée sur 0.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
corrs_cible = train_X.corrwith(...).dropna().sort_values(...)
corrs_cible_extremes = pandas.concat([...]).rename("SalePrice")
corrs_variables = train_X[corrs_cible_extremes.index].corr()
corrs = pandas.concat([corrs_cible_extremes, corrs_variables], axis=1)

_, ax = plt.subplots(figsize=(10, 6))
seaborn.heatmap(corrs, vmin=..., vmax=..., cmap="vlag", ax=ax)
plt.show()
```

#### Solution

In [ ]:
corrs_cible = (
  train_X
  # Calcul des corrélations avec la variable de sortie
  .corrwith(train_y)
  # Suppression des NaN créés : ils correspondent aux colonnes
  # constantes
  .dropna()
  # Tri par valeur de corrélation : les colonnes les plus corrélées
  # seront les premières, les plus anti-corrélées les dernières
  .sort_values(ascending=False)
)

# Récupération des dix premières et dix dernières valeurs
corrs_cible_extremes = pandas.concat([corrs_cible[:10], corrs_cible[-10:]])
corrs_cible_extremes = corrs_cible_extremes.rename("SalePrice")

# Calcul des corrélations des variables les plus corrélées (et anti-corrélées) entre elles
corrs_variables = train_X.loc[:, corrs_cible_extremes.index].corr()

# Regroupement des corrélations avec la cible et des corrélations entre
# variables
corrs = pandas.concat([corrs_cible_extremes, corrs_variables], axis=1)

# Affichage de la matrice de corrélation à l'aide d'une heatmap
_, ax = plt.subplots(figsize=(10, 6))
seaborn.heatmap(corrs, vmin=-1, vmax=1, cmap=seaborn.diverging_palette(220, 20, n=100))
ax.set_title("Plus grandes corrélations et anti-corrélations")
plt.show()

print(corrs_cible_extremes.round(2))

La qualité générale (`OverallQual`, 0,79) et la surface habitable (`GrLivArea`, 0,71) sont les plus corrélées au prix, suivies d'autres variables de qualité (extérieur, cuisine, sous-sol) et de taille (garage, sous-sol, premier étage). Plusieurs de ces variables sont aussi très corrélées entre elles (`GarageCars` et `GarageArea`, `TotalBsmtSF` et `1stFlrSF`) : elles portent en partie la même information. Les anti-corrélations, plus faibles, désignent des maisons plus modestes (garage non fini ou détaché, fondations en parpaing).

### Exploration des variables les plus corrélées

Les variables les plus corrélées sont maintenant connues. Il reste à nous renseigner sur ces dernières.

- *Graphez la distribution jointe de `GrLivArea` et `SalePrice`, ainsi que la distribution de `SalePrice` pour chaque valeur de `OverallQual`.*
- *Que remarquez-vous sur ces deux graphiques ?*

**Documentation utile**

- [`seaborn.jointplot`](https://seaborn.pydata.org/generated/seaborn.jointplot.html)
- [`seaborn.violinplot`](https://seaborn.pydata.org/generated/seaborn.violinplot.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `seaborn.jointplot(x=train_X["GrLivArea"], y=train_y)` : le nuage de points des deux variables, avec la distribution de chacune sur les bords.
- `seaborn.violinplot(x=train_X["OverallQual"], y=train_y)` : un violon (la densité de `SalePrice`) par valeur de `OverallQual`.
- Regardez les points qui s'écartent du nuage, et ce que `maisons` dit de ces ventes (`SaleCondition`).

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
seaborn.jointplot(x=..., y=train_y)
plt.show()
seaborn.violinplot(x=..., y=train_y)
plt.show()
```

#### Solution

In [ ]:
seaborn.jointplot(x=train_X["GrLivArea"], y=train_y)
plt.show()
seaborn.violinplot(x=train_X["OverallQual"], y=train_y)
plt.show()

print(
  maisons.loc[maisons["GrLivArea"] > 4000, ["GrLivArea", "SalePrice", "SaleCondition"]]
)

Le prix croît avec la surface habitable et avec la qualité générale, et sa dispersion augmente avec elles : les maisons de qualité 9 et 10 ont des prix bien plus variés que celles de qualité 4.

Deux ventes sortent nettement du nuage : les deux plus grandes surfaces (plus de 4 500 sq ft), vendues moins de 200 000 $. Sont-ce des erreurs dans le jeu de données, ou des maisons aux caractéristiques inhabituelles ? `maisons` donne leur condition de vente : `Partial`, une maison pas encore terminée lors de la vente. L'article de De Cock, l'auteur du dataset, recommande de retirer toutes les maisons de plus de 4 000 sq ft avant d'entraîner un modèle : ces deux-là, et deux ventes à environ 750 000 $.

## Appliquer ce que l'exploration a appris

L'exploration ne sert que si elle change la préparation des données. Appliquez ses deux constats aux données d'entraînement :

- *Retirez de `train_X` et `train_y` les ventes de plus de 4 000 sq ft de `GrLivArea`, comme le recommande De Cock. Combien en retirez-vous ?*
- *Passez la cible au logarithme avec `numpy.log1p`.*
- *Nommez les résultats `train_X_propre` et `train_y_log_propre`, puis comparez la corrélation de `GrLivArea` avec la cible avant et après, et retracez leur distribution jointe.*
- *Pourquoi ne retire-t-on aucune maison de `test_X` ?*

**Documentation utile**

- [`pandas.Series.corr`](https://pandas.pydata.org/docs/reference/api/pandas.Series.corr.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `garde = train_X["GrLivArea"] <= 4000` : une `Series` de booléens ; `train_X[garde]` et `train_y[garde]` gardent les mêmes lignes, et `(~garde).sum()` compte celles qu'on retire.
- `s1.corr(s2)` : la corrélation de deux `Series` de même index.
- Attendu : 4 ventes retirées ; une corrélation d'environ 0,71 avant, 0,72 après.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
garde = ...
print("Ventes retirées :", ...)

train_X_propre = train_X[garde]
train_y_log_propre = numpy.log1p(...)

print("Avant :", train_X["GrLivArea"].corr(train_y))
print("Après :", train_X_propre["GrLivArea"].corr(...))

seaborn.jointplot(x=..., y=...)
plt.show()
```

### Solution

In [ ]:
garde = train_X["GrLivArea"] <= 4000
print("Ventes retirées :", (~garde).sum())

train_X_propre = train_X[garde]
train_y_log_propre = numpy.log1p(train_y[garde])

print(f"Corrélation avant : {train_X['GrLivArea'].corr(train_y):.3f}")
print(f"Corrélation après : {train_X_propre['GrLivArea'].corr(train_y_log_propre):.3f}")
print(f"Asymétrie de la cible après : {train_y_log_propre.skew():.2f}")
print(
  "Maisons de plus de 4 000 sq ft dans test_X :", (test_X["GrLivArea"] > 4000).sum()
)

seaborn.jointplot(x=train_X_propre["GrLivArea"], y=train_y_log_propre)
plt.show()

On retire 4 ventes : les deux ventes partielles, et les deux ventes à environ 750 000 $, qui suivaient pourtant la tendance (De Cock retire toutes les maisons de plus de 4 000 sq ft, pour la simplicité). La corrélation de `GrLivArea` avec la cible passe de 0,709 à 0,719, et l'asymétrie de la cible tombe à 0,07 : le nuage ne contient plus de point isolé, et sa tendance est plus régulière.

On ne retire rien de `test_X` : il représente les maisons qu'on devra estimer en production, et l'on ne choisit pas celles-là. Une maison de plus de 4 000 sq ft s'y trouve : sa prédiction sera peu fiable, puisque le modèle n'en aura vu aucune ; on peut le signaler à qui l'utilise.